# LLM & GenAI Pipeline — Phase 1
**Author:** Shruti  
**Description:** A document-aware LLM pipeline built from scratch. Covers JSON handling, API calls, prompt engineering, file reading, pandas, error handling, and a final mini project.

**Stack:** Python, Groq API (LLaMA 3.1), Pandas

---

## Setup
Install libraries and configure the API client.

In [ ]:
!pip install groq pandas -q

import json
import pandas as pd
from groq import Groq

# Replace with your Groq API key from console.groq.com
client = Groq(api_key="your-key-here")

---
## Day 1 — JSON & First API Call

**Skills practiced:** Creating and reading JSON files, working with dicts and lists, making the first LLM API call.

### Exercise 1 — Create and save a JSON file
Create a list of dicts representing LLM responses and save to a JSON file.

In [ ]:
responses = [
    {"id": 1, "model": "gpt-4", "prompt": "What is a gpt-4 model"},
    {"id": 2, "model": "claude-3", "prompt": "What is a claude-3 model"},
    {"id": 3, "model": "gpt-4", "prompt": "How to use a gpt-4 model"},
]

with open("responses.json", "w") as f:
    json.dump(responses, f, indent=2)

print("Saved!")

### Exercise 2 — Read JSON and print fields
Read the JSON file back and print the model and prompt for each item.

In [ ]:
with open("responses.json", "r") as f:
    responses = json.load(f)

for n in responses:
    print(n["model"], n["prompt"])

### Exercise 3 — Filter by model and count
Filter responses by model name and count how many match.

In [ ]:
count = 0
for n in responses:
    if n["model"] == "gpt-4":
        count += 1
        print(n["prompt"])

print(f"Total gpt-4 responses: {count}")

### Exercise 4 & 5 — First API call + Loop and save responses
Call the Groq API for each prompt and save all responses to a JSON file.

In [ ]:
prompts = [
    "What is Retrieval-Augmented Generation in AI? Explain in 2 sentences.",
    "What are embeddings in AI? Explain in 2 sentences.",
    "What is fine-tuning? Explain in 2 sentences."
]

results = []
for prompt in prompts:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    results.append({"prompt": prompt, "response": response.choices[0].message.content})
    print(f"Done: {prompt[:50]}...")

with open("llm_responses.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved!")

---
## Day 2 — Prompt Templates & Multi-turn Conversations

**Skills practiced:** Simulating chat history, filtering by role, writing reusable prompt functions, dynamic API calls.

### Exercise 1 & 2 — Simulate chat history and filter by role
Create a list of chat messages alternating between user and assistant, then filter and count user messages.

In [ ]:
chat_history = [
    {"role": "user", "content": "What is RAG?"},
    {"role": "assistant", "content": "RAG stands for Retrieval-Augmented Generation..."},
    {"role": "user", "content": "Can you give me an example?"},
    {"role": "assistant", "content": "Sure! Imagine a chatbot that searches documents..."},
    {"role": "user", "content": "How is it different from fine-tuning?"},
]

count = 0
for n in chat_history:
    if n["role"] == "user":
        count += 1
        print(f"User: {n['content']}")

print(f"\nTotal user messages: {count}")

### Exercise 3, 4 & 5 — Reusable prompt function + API call + Loop and save
Build a `build_prompt` function that creates messages dynamically, then use it to loop through questions and save results.

In [ ]:
def build_prompt(system_instruction, user_question):
    return [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_question},
    ]

questions = ["What is RAG?", "What are embeddings?", "What is fine-tuning?"]
results = []

for q in questions:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=build_prompt("You are a helpful AI assistant.", q)
    )
    results.append({"question": q, "response": response.choices[0].message.content})
    print(f"Done: {q}")

with open("day2_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved!")

---
## Day 3 — File Reading & Context-Aware Prompts

**Skills practiced:** Reading text files, splitting and cleaning text, feeding document context to an LLM.

### Exercise 1 & 2 — Create knowledge base, read and clean sentences
Write a knowledge base to a text file, read it back, split into clean sentences.

In [ ]:
text = """Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation.
It works by first retrieving relevant documents from a knowledge base, then using them as context for the LLM.
RAG helps reduce hallucinations by grounding the model in real data.
Embeddings are vector representations of text that capture semantic meaning.
Similar texts have similar embeddings and are close together in vector space.
Fine-tuning adapts a pre-trained model on specific data to improve performance on a task."""

with open("knowledge.txt", "w") as f:
    f.write(text)

with open("knowledge.txt", "r") as f:
    text = f.read()

sentences = [s.strip() for s in text.split(".") if s.strip() != ""]

for i, sentence in enumerate(sentences):
    print(f"{i}: {sentence}")

### Exercise 3, 4 & 5 — Feed context to LLM for each sentence
Use each sentence as context in the system prompt and ask the LLM to explain it in simpler terms.

In [ ]:
results = []

for sentence in sentences:
    system_instruction = f"You are a helpful assistant. Use this context to answer: {sentence}"
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=build_prompt(system_instruction, "Can you explain this in simpler terms?")
    )
    results.append({"sentence": sentence, "response": response.choices[0].message.content})
    print(f"Done: {sentence[:50]}...")

with open("day3_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved!")

---
## Day 4 — Pandas & CSV Pipeline

**Skills practiced:** Loading CSV files with pandas, converting to dicts, filtering dataframes, feeding structured data to an LLM.

### Exercise 1, 2 & 3 — Create CSV, load with pandas, filter
Create a topics CSV, load it with pandas, explore it, and filter rows.

In [ ]:
data = {
    "id": [1, 2, 3, 4, 5],
    "topic": ["RAG", "Embeddings", "Fine-tuning", "Transformers", "Prompt Engineering"],
    "description": [
        "A technique combining retrieval with generation",
        "Vector representations of text",
        "Adapting a pre-trained model on specific data",
        "Architecture behind modern LLMs",
        "Crafting inputs to get better LLM outputs"
    ]
}

df = pd.DataFrame(data)
df.to_csv("topics.csv", index=False)

df = pd.read_csv("topics.csv")
print(df.head(3))
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

filtered = df[df["topic"].isin(["RAG", "Embeddings"])]
print("\nFiltered:")
print(filtered)

### Exercise 4 & 5 — CSV to LLM pipeline
Loop through CSV rows, feed each description as context to the LLM, and save results.

In [ ]:
records = df.to_dict("records")
results = []

for record in records:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=build_prompt(
            f"You are a helpful assistant. Use this context: {record['description']}",
            "Can you explain this in simpler terms?"
        )
    )
    results.append({"topic": record["topic"], "response": response.choices[0].message.content})
    print(f"Done: {record['topic']}")

with open("day4_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} results!")

---
## Day 5 — Error Handling & Retry Logic

**Skills practiced:** try/except blocks, capturing error messages, retry logic for robust API calls.

### Exercise 1, 2, 3 & 4 — Safe API call with retry logic
Wrap the API call in a function with try/except and automatic retries.

In [ ]:
def safe_api_call(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
    return "All retries failed"

# Test with valid prompt
print(safe_api_call("What is RAG in 2 sentences?"))

### Exercise 5 — Full CSV pipeline with error handling
Combine pandas, safe API calls, and JSON saving into one robust pipeline.

In [ ]:
records = pd.read_csv("topics.csv").to_dict("records")
results = []

for record in records:
    result = safe_api_call(f"Explain {record['topic']}: {record['description']}")
    results.append({"topic": record["topic"], "response": result})
    print(f"Done: {record['topic']}")

with open("day5_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} results!")

---
## Day 6 — Mini Project: Document-Aware LLM Pipeline

**Skills practiced:** Combining all Phase 1 skills into one complete, production-ready pipeline.

**What this does:**
1. Reads a knowledge base from a text file
2. Loads structured topic data from CSV
3. Matches each topic to its relevant sentence in the knowledge base
4. Calls the LLM with context-aware prompts and retry logic
5. Saves all results to a JSON file

In [ ]:
# ── Phase 1 Mini Project: Document-Aware LLM Pipeline ──

# Step 1: Read knowledge base
with open("knowledge.txt", "r") as f:
    text = f.read()

sentences = [s.strip() for s in text.split(".") if s.strip() != ""]

# Step 2: Load topics from CSV
records = pd.read_csv("topics.csv").to_dict("records")

# Step 3: Match context, call API, collect results
results = []
for record in records:
    # Find matching sentence from knowledge base
    context = ""
    for sentence in sentences:
        if record["topic"] in sentence:
            context = sentence
            break

    # Call LLM with context
    result = safe_api_call(
        f"Use this context: {context}. Explain {record['topic']} in simple terms."
    )
    results.append({"topic": record["topic"], "context": context, "response": result})
    print(f"Done: {record['topic']}")

# Step 4: Save results
with open("final_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nPipeline complete! Saved {len(results)} results to final_results.json")